# ASTRA-GUARD: Phase 2 — Spacecraft Health AI Exploration
**AI-Powered Integrated Space Mission Safety & Intelligence**
Team OMEN · AITHON 2.0 · Track 11 Aerospace, Telemetry & SpaceTech

This notebook demonstrates:
1. **Preprocessing & Robust Scaling:** Scaling multi-channel telemetry using median and IQR fitted exclusively on nominal data (rules.md §5).
2. **PyTorch LSTM Autoencoder:** Sequence-to-sequence reconstruction of multi-channel LEO orbital patterns ($W=64$, $\Delta t=10\text{s}$).
3. **Score Calibration & Persistence Filter:** Monotonic mapping to $[0, 1]$ bounds and $k$-of-$n$ filtering to suppress spurious false alarms.
4. **Channel-Level Explainability:** Attribution breakdown revealing root-cause subsystems (e.g., ADCS reaction wheel thermal runaway).
5. **Baseline Comparison:** Benchmarking against an engineered Isolation Forest baseline with event-wise metrics and false alarms/day.


In [1]:
import os
import sys

sys.path.insert(0, os.path.abspath('..'))
from data.synthetic.telemetry_gen import generate_telemetry_series
from ml.health.module import HealthRiskModule

print('Environment and Health AI modules loaded successfully.')

## 1. Train Health AI Module on Nominal Orbit Telemetry
Per `rules.md §5`, all scalers and anomaly thresholds are fitted strictly on nominal training telemetry.

In [2]:
df_nom = generate_telemetry_series(asset_id='SENTINEL-2A', duration_minutes=120, step_seconds=10, fault_type=None, seed=42)
module = HealthRiskModule(window_size=64, window_step=1)
module.fit(df_nom)
print(f'Active Engine: {module.active_engine}')
print(f'LSTM Calibrator Anchors: p50={module.lstm_model.calibrator.p50:.6f}, p95={module.lstm_model.calibrator.p95:.6f}, p99={module.lstm_model.calibrator.p99:.6f}')
print(f'IsoForest Calibrator Anchors: p50={module.isoforest_model.calibrator.p50:.6f}, p95={module.isoforest_model.calibrator.p95:.6f}, p99={module.isoforest_model.calibrator.p99:.6f}')

## 2. Injected Fault Evaluation & Root Cause Attribution
Evaluate an injected reaction wheel thermal drift fault starting at T+45m.

In [3]:
df_drift = generate_telemetry_series(asset_id='SENTINEL-2A', duration_minutes=90, step_seconds=10, fault_type='drift', fault_start_minute=45, fault_channel='wheel_temp', seed=99)
res = module.score(df_drift.iloc[-64*7:], asset_id='SENTINEL-2A')
print('ModuleResult Contract Output:')
print(f'  Risk Score (r_h): {res.score} (0..1 scale)')
print(f'  Confidence:       {res.confidence}')
print(f'  Sustained Alert:  {res.evidence.get("is_sustained")}')
print(f'  Model Type:       {res.evidence.get("model_type")}')

# Produce attributions
attributions = module.explain(df_drift.iloc[-64*7:], res)
print('\nRoot-Cause Channel Attributions:')
for attr in attributions:
    print(f'  - {attr.driver:12s} contribution: {attr.contribution * 100:.1f}% ({attr.direction})')

## 3. Benchmark Verification: LSTM AE vs. Isolation Forest Baseline
Load results from `evaluation/results/health.json`.

In [4]:
import json

with open('../evaluation/results/health.json', 'r') as f:
    results = json.load(f)

print('Model Comparison Summary:')
print(f'  Primary (LSTM AE):    Event F1 = {results["primary_model"]["metrics"]["event_f1"]:.4f} | False Alarms/Day = {results["primary_model"]["metrics"]["false_alarms_per_day"]:.2f}')
print(f'  Baseline (IsoForest): Event F1 = {results["baseline_model"]["metrics"]["event_f1"]:.4f} | False Alarms/Day = {results["baseline_model"]["metrics"]["false_alarms_per_day"]:.2f}')
print(f'  Improvement Delta:    +{results["comparison_delta"]["f1_improvement"]:.4f} F1 | -{results["comparison_delta"]["false_alarms_reduction_per_day"]:.2f} False Alarms/Day')